In [ ]:
# Import libraries and set desired options
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.sparse import csr_matrix, hstack
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

sns.set()
%config InlineBackend.figure_format = 'retina'

$\def\m#1{\mathbf{#1}}$
$\def\mm#1{\boldsymbol{#1}}$
$\def\mb#1{\mathbb{#1}}$
$\def\c#1{\mathcal{#1}}$
$\def\mr#1{\mathrm{#1}}$
$\newenvironment{rmat}{\left[\begin{array}{rrrrrrrrrrrrr}}{\end{array}\right]}$
$\newcommand\brm{\begin{rmat}}$
$\newcommand\erm{\end{rmat}}$
$\newenvironment{cmat}{\left[\begin{array}{ccccccccc}}{\end{array}\right]}$
$\newcommand\bcm{\begin{cmat}}$
$\newcommand\ecm{\end{cmat}}$

# Homework 2
## Homework guideline
- The deadline is Feb 21st 10:30am. Submission after the deadline will not be graded.

- Submit your work(your reasoning and your code) as a SINGLE .ipynb document. Please rename the document as _HW1_YOURNAME.ipynb_ (for example, _HW1_FELIX.ipynb_). You are responsible for checking that you have correctly submitted the correct document. If your code cannot run, you may receive NO point.

- Please justify all short answers with a brief explanation.

- You only use the Python packages included in the following cell. You are not allowed to use other advanced package or modules unless you are permitted to.

- In your final submission include the plots produced by the unedited code as presented below, as well as any additional plots produced after editing the code during the course of a problem. You may find it necessary to copy/paste relevant code into additional cells to accomplish this.

- Feel free to use the lecture notes and other resources. But you
must understand, write, and hand in your own answers. In addition, you must write and submit
your own code in the programming part of the assignment (we may run your code).
If you copy someone else homework solution, both of you may receive ZERO point.


- **You must write your own code and fill in the your answer in the text box.** If you fail to do either of that, you will receive zero point.

- **If you copy and paste the code from chatGPT, Bard or etc, you will receive zero point**.



---



---




Today we will reproduce a couple of baselines in "Intruder Detection through Webpage Session Tracking"(a.k.a. "Alice") and discuss ideas for further improvement. You'll sense the spirit of realistic datasets: brainstorming model or validation scheme improvements, adding new features, implementing new ideas in code, climbing the leaderboard – that's exciting! And that's very useful as well, especially when you're only starting your ML journey. In this assignment, we'll also work with sparse matrices (which is often the only way to work with some types of data, from the computation perspective), train Logistic Regression models, and practice feature engineering.

We have discussed in class the competition with various different methods.

In this homework, we will work through to beat these baseline. At the end, you need to come up with good features to beat leadboard!

**Problem description**

In this competition, we'll analyze the sequence of websites consequently visited by a particular person and try to predict whether this person is Alice or someone else. As a metric we will use [ROC AUC](https://en.wikipedia.org/wiki/Receiver_operating_characteristic).

### Data Download
First, read the training and test sets. Then we'll explore the data in hand and do a couple of simple exercises.


**Data access:** the data is stored in https://drive.google.com/file/d/1WvBuXCOMho1ZAWcJLi4X-a1l-ZReG-5K/view?usp=sharing.

**Load Data:** Once you open the link in the brower, make sure you click the "Add shortcut to Drive" and now your google drive should show up the zip file.  Then you run the following code to link colab to your google drive.

### Data Description
The training data set contains the following features:

- **site1** – id of the first visited website in the session
- **time1** – visiting time for the first website in the session
- ...
- **site10** – id of the tenth visited website in the session
- **time10** – visiting time for the tenth website in the session
- **target** – target variable, 1 for Alice's sessions, and 0 for the other users' sessions
    
User sessions are chosen in the way that they are shorter than 30 min. long and contain no more than 10 websites. I.e. a session is considered over either if a user has visited 10 websites or if a session has lasted over 30 minutes.

There are some empty values in the table, it means that some sessions contain less than ten websites. Replace empty values with 0 and change columns types to integer. Also load the websites dictionary and check how it looks like:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir("/content/drive/My Drive")

In [ ]:
!unzip catch-me-if-you-can-intruder-detection-through-webpage-session-tracking2.zip

In [ ]:
times = ["time%s" % i for i in range(1, 11)]
# customize the paths if needed
train_df = pd.read_csv("train_sessions.csv", index_col="session_id", parse_dates=times)
test_df = pd.read_csv("test_sessions.csv", index_col="session_id", parse_dates=times)

# Sort the data by time
train_df = train_df.sort_values(by="time1")

# Look at the first rows of the training set
train_df.head(2)

In [ ]:
# Change site1, ..., site10 columns type to integer and fill NA-values with zeros
sites = ["site%s" % i for i in range(1, 11)]
train_df[sites] = train_df[sites].fillna(0).astype(np.uint16)
test_df[sites] = test_df[sites].fillna(0).astype(np.uint16)

# Load websites dictionary
with open("site_dic.pkl", "rb") as input_file:
    site_dict = pickle.load(input_file)

# Create dataframe for the dictionary
sites_dict = pd.DataFrame(
    list(site_dict.keys()), index=list(site_dict.values()), columns=["site"]
)
print(u"Websites total:", sites_dict.shape[0])
sites_dict.head()


Before we start training models, we have to perform Exploratory Data Analysis ([EDA](https://en.wikipedia.org/wiki/Exploratory_data_analysis)). Today, we are going to perform a shorter version, but we will use other techniques as we move forward. Let's check which websites in the training data set are the most visited. As you can see, they are Google services and a bioinformatics website (a website with 'zero'-index is our missed values, just ignore it):

In [ ]:
# Top websites in the training data set
top_sites = (
    pd.Series(train_df[sites].values.flatten())
    .value_counts()
    .sort_values(ascending=False)
    .head(5)
)
print(top_sites)
sites_dict.loc[top_sites.drop(0).index]



---


Now let us look at the timestamps and try to characterize sessions as timeframes:

In [ ]:
# Create a separate dataframe where we will work with timestamps
time_df = pd.DataFrame(index=train_df.index)
time_df["target"] = train_df["target"]

# Find sessions' starting and ending
time_df["min"] = train_df[times].min(axis=1)
time_df["max"] = train_df[times].max(axis=1)

# Calculate sessions' duration in seconds
time_df["seconds"] = (time_df["max"] - time_df["min"]) / np.timedelta64(1, "s")

time_df.head()


In the next question, we are using the notion of "approximately the same". To be strict, let's define it: $a$ is approximately the same as $b$ ($a \approx b $) if their difference is less than or equal to 5% of the maximum between $a$ and $b$, i.e. $a \approx b \leftrightarrow \frac{|a-b|}{max(a,b)} \leq 0.05$.



---



---
# Q1:  User Identification with Logistic Regression (60pt)

## Q1.1: Brief Exploratory Data Analysis (10pt)
- What kind of websites does Alice visit the most?

- On average, is Alice's session shorter than that of other users

- What percentage of all sessions belong to Alice?

- Are minimum and maximum durations of Alice's and other users' sessions approximately the same?

- Is standard deviation of Alice's sessions duration approximately the same as for non-Alice's sessions




In [ ]:
# your code starts here
# Alice values
alice = train_df[train_df['target'] == 1]
# Non Alice values
non_alice = train_df[train_df['target'] == 0]
# extracting sites
alice_sites = alice[sites].values.flatten()
# counting unique values count
sites_values = pd.Series(alice_sites).value_counts()
# extracting top 5 values
top_sites = sites_values.head(5)
# adjusting it with website names
print(sites_dict.loc[top_sites.index])

In [ ]:
# Alice values
alice_time = time_df[time_df['target'] == 1]
# Non Alice values
non_alice_time = time_df[time_df['target'] == 0]
# Taking mean of alice time
alice_average =  alice_time['seconds'].mean()
# Taking mean of non alice time
non_alice_average = non_alice_time['seconds'].mean()
# Printing Both values
print("Alice Average session duration in Seconds:", alice_average, "seconds")
print("non-Alice users Average session duration in Seconds:", non_alice_average, "seconds")

In [ ]:
# Total dataset values
total_data = len(train_df)
# Total alice values in dataset
alice_data = len(alice)
# Finding Percentage of alice
percentage = (alice_data / total_data) * 100
# Printing value
print('Alice session Percentage', percentage , '%')

In [ ]:
# Min values for alice and non alice
alice_min = time_df[time_df['target'] == 1]['seconds'].min()
non_alice_min = time_df[time_df['target'] == 0]['seconds'].min()
# Max values for alice and non alice
alice_max = time_df[time_df['target'] == 1]['seconds'].max()
non_alice_max = time_df[time_df['target'] == 0]['seconds'].max()
# Finding difference between both
min_diff = abs(alice_min - non_alice_min) / max(alice_min, non_alice_min)
max_diff = abs(alice_max - non_alice_max) / max(alice_max, non_alice_max)
# Print if difference is less than 0.5 than it is similar
print("Is minimum duration approximately the same?", min_diff <= 0.05)
print("Is maximum duration approximately the same?", max_diff <= 0.05)

In [ ]:
# Finding std of both type of user
alice_std = time_df[time_df['target'] == 1]['seconds'].std()
non_alice_std = time_df[time_df['target'] == 0]['seconds'].std()
# Finding difference between both
std_diff = abs(alice_std - non_alice_std) / max(alice_std, non_alice_std)
# Printing difference and is they are same or Not
print('standard deviation of sessios' , std_diff)
print("Is standard deviation of session duration approximately the same?", std_diff <= 0.05)

### **Discuss your findings**:

**a**

Top 5 most visited websites by alice are:

1.   i1.ytimg.com
2.   s.youtube.com
3.   www.youtube.com
4.   www.facebook.com
5.   www.google.fr

**b:**

The duration of non Alice User are higher than Alice user. The Non alice user spend 139.28237232552215 seconds and Alice user spend 52.29647366129734 seconds.

**c**

The percentage of Alice user are 0.905896411514389 % means 0.9 percent user are alice in whole dataset

**d**

The finding shows that Max values are approximately the same, where by Min values are not approximately the same.

**e**

The std is not approximately the same as the difference show is more than 0.05. The difference is are 0.48.






---



---


In order to train our first model, we need to prepare the data. First of all, exclude the target variable from the training set. Now both training and test sets have the same number of columns, therefore aggregate them into one dataframe.  Thus, all transformations will be performed simultaneously on both training and test data sets.

On the one hand, it leads to the fact that both data sets have one feature space (you don't have to worry that you forgot to transform a feature in some data sets). On the other hand, processing time will increase.
For the enormously large sets it might turn out that it is impossible to transform both data sets simultaneously (and sometimes you have to split your transformations into several stages only for train/test data set).
In our case, with this particular data set, we are going to perform all the transformations for the whole united dataframe at once, and before training the model or making predictions we will just take its appropriate part.

In [ ]:
# Our target variable
y_train = train_df["target"]

# United dataframe of the initial data
full_df = pd.concat([train_df.drop("target", axis=1), test_df])

# Index to split the training and test data sets
idx_split = train_df.shape[0]

For the very basic model, we will use only the visited websites in the session (but we will not take into account timestamp features). The point behind this data selection is: *Alice has her favorite sites, and the more often you see these sites in the session, the higher probability that this is Alice's session, and vice versa.*

Let us prepare the data, we will take only features `site1, site2, ... , site10` from the whole dataframe. Keep in mind that the missing values are replaced with zero. Here is how the first rows of the dataframe look like:

In [ ]:
# Dataframe with indices of visited websites in session
full_sites = full_df[sites]
full_sites.head()

Sessions are sequences of website indices, and data in this representation is useless for machine learning method (just think, what happens if we switched all ids of all websites).


---



---


## Q1.2: Data Preparation (10pt)
According to our hypothesis (Alice has favorite websites), we need to transform this dataframe so each website has a corresponding feature (column) and its value is equal to number of this website visits in the session.

For the work with such matrices you can use `csr_matrix` function, check [documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html) to understand what possible types of sparse matrices are, how to work with them and in which cases their usage is most effective.

In [ ]:
# All sites mark as one
one_data = [1] * full_sites.size
# Making data flatten to get columns values
column_values = full_sites.values.ravel()
# Making csr matrix with row values
row_values = np.arange(0, full_sites.shape[0] * full_sites.shape[1] + 1, full_sites.shape[1])
# Adding 1 to make shape compitable
data_shape = (full_sites.shape[0], max(full_sites.values.max(), 10) + 1)
# Create the sparse matrix using Compressed Sparse Row (CSR) format
sprase = csr_matrix((one_data, column_values, row_values), shape=data_shape)



---





---



## Q1.3: First Try: Logistic Regression (10pt)
Let us build our first model, using [logistic regression](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) implementation from ` Sklearn` with default parameters. We will use the first 90% of the data for training (the training data set is sorted by time), and the remaining 10% for validation.

- Let's write a simple function that returns the quality of the model and then train our first classifier. What is the result on the validation set? You will need to use it in the later questions.

- To make a prediction on the test data set **we need to train the model again on the entire training data set** (until this moment, our model used only part of the data for training), which will increase its generalizing ability. You don't need to calculate the score for testing set since there is no target value.


In [ ]:
def get_auc_lr_valid(X, y, C=1.0, seed=17, ratio=0.9):
  # Data splitting
  split = int(X.shape[0] * ratio)
  X_train, X_valid = X[:split], X[split:]
  y_train, y_valid = y[:split], y[split:]
  # Making model and fitting model
  model = LogisticRegression(C=C, random_state=seed, solver='liblinear')
  model.fit(X_train, y_train)
  # Taking proba from model
  y_proba = model.predict_proba(X_valid)[:, 1]
  # Calucating roc auc score from proba
  auc_score = roc_auc_score(y_valid, y_proba)
  return auc_score

In [ ]:
# your code on training
auc_score = get_auc_lr_valid(sprase[:idx_split], y_train, C=1.0, seed=17, ratio=0.9)
print("Auc Roc score on Valid Data", auc_score)

### **Discuss your findings**:

The Auc roc score getting on validation data is 0.9157530199778038 which is good indication that model is good in classification of labels.




---



---


## Q1.4: Second Try: Feature Engineering (10pt)
Now we are going to improve the quality of our model by engineering new features.

- Create a feature that will be a number in YYYYMM format from the date when the session was held, for example 201407 -- year 2014 and 7th month.

- Plot the graph of the number of Alice sessions versus the new feature, start_month. Discuss the result.

- In this way, we have an illustration and thoughts about the usefulness of the new feature, add it to the training sample and check the quality of the new model. What is your score on the validation set? You should find the quality of the model has decreased significantly.

- You need to apply the standarization on this new feature and train the model again. What is your score on the validation set this time?

**Hint:** the graph will be more explicit if you treat `start_month` as a categorical ordinal variable

In [ ]:
# Convert strat month data into new feature name
full_df['start_month'] = full_df['time1'].apply(lambda x: 100 * x.year + x.month)
# Plotting plot of new feature and alice (label)
plt.figure(figsize=(15, 6))
sns.countplot(data=full_df[:idx_split], x='start_month', hue=y_train)
# Title of plot
plt.title('Number of Alice Sessions vs. Start Month')
#X and y axis name of plot
plt.xlabel('Year-Month')
plt.ylabel('Number of Sessions')
plt.show()

In [ ]:
# Concatenate the new feature with the existing sparse matrix
start_month_sparse = csr_matrix(pd.get_dummies(full_df['start_month']))
full_sparse = hstack([sprase, start_month_sparse]).tocsr()

# Calculate AUC ROC score for the model with the new feature added
auc_score_new_feature = get_auc_lr_valid(full_sparse[:idx_split], y_train, C=1.0, seed=17, ratio=0.9)
print("AUC ROC score on the validation data including the new feature:", auc_score_new_feature)

In [ ]:
# Standardize the new feature
scaler = StandardScaler()
start_month_scaled = scaler.fit_transform(full_df[['start_month']])
start_month_scaled_sparse = csr_matrix(start_month_scaled)
# Concatenate the standardized new feature with the existing sparse matrix
full_sparse_scaled = hstack([sprase, start_month_scaled_sparse]).tocsr()
# Calculate AUC ROC score for the model with the standardized feature added
auc_score_scaled_feature = get_auc_lr_valid(full_sparse_scaled[:idx_split], y_train, C=1.0, seed=17, ratio=0.9)
print("AUC ROC score on the validation data including the standardized feature:", auc_score_scaled_feature)

### **Discuss your findings:**
The result shows that Model score have increased to 0.92, the reason for this could be this new feature is bit compitable to predict the Label. However when we scaled it its accuracy decline.




---



---


## Q1.5: Third Try: Add more features? (10pt)
- Add to the training set a new feature "n_unique_sites" – the number of the unique web-sites in a session. Calculate how the quality on the validation set has changed. Shall we add this feature?

- Add two new features: start_hour and morning. Calculate the metric. Which of these features gives an improvement.

The `start_hour` feature is the hour at which the session started (from 0 to 23), and the binary feature `morning` is equal to 1 if the session started in the morning and 0 if the session started later (we assume that morning means `start_hour` is equal to 11 or less).

Will you scale the new features? Make your assumptions and test them in practice.

In [ ]:
# Standardize the n_unique_sites feature
n_unique_sites_scaled = scaler.fit_transform(full_df[['n_unique_sites']])
n_unique_sites_scaled_sparse = csr_matrix(n_unique_sites_scaled)

# Concatenate the standardized n_unique_sites feature with the existing sparse matrix
full_sparse_scaled_with_n_unique_sites = hstack([full_sparse_scaled, n_unique_sites_scaled_sparse]).tocsr()

# Calculate AUC ROC score for the model with the standardized n_unique_sites feature added
auc_score_with_n_unique_sites = get_auc_lr_valid(full_sparse_scaled_with_n_unique_sites[:idx_split], y_train, C=1.0, seed=17, ratio=0.9)
print("AUC ROC score on the validation data including the standardized n_unique_sites feature:", auc_score_with_n_unique_sites)

 The AUC ROC score was 0.92. After adding and scaling the "n_unique_sites" feature alone, the score dropped to 0.60. This indicates that the "n_unique_sites" feature alone does not contribute much to the model's performance.

In [ ]:
# Add start_hour and morning features
full_df['start_hour'] = full_df['time1'].dt.hour
full_df['morning'] = (full_df['start_hour'] <= 11).astype(int)
# Scale start_hour and morning features together
full_df[['start_hour_scaled', 'morning_scaled']] = scaler.fit_transform(full_df[['start_hour', 'morning']])
# Create a new sparse matrix with the scaled start_hour and morning features
new_features_sparse = csr_matrix(full_df[['start_hour_scaled', 'morning_scaled']].values)
# Concatenate the new features with the existing sparse matrix
full_sparse_with_new_features = hstack([full_sparse_scaled_with_n_unique_sites, new_features_sparse]).tocsr()
X_train_with_new_features = full_sparse_with_new_features[:idx_split]
# Calculate AUC ROC score for the model with the new features added
auc_score_all_scaled = get_auc_lr_valid(X_train_with_new_features, y_train, C=1.0, seed=17, ratio=0.9)
# Print result
print("AUC ROC score with scaled n_unique_sites and other features:", auc_score_all_scaled)

After adding the "n_unique_sites" feature and scaling it, along with the "start_hour" and "morning" features, the AUC ROC score improved to 0.95. This indicates that collectively, these features contribute positively to the model's performance, resulting in a higher accuracy compared to using only the "n_unique_sites" feature.

### **Discuss your findings:**

By adding n_unique feature the score is around 0.90 which still not better to baseline model. Whereby adding all three variables the score increase to 0.95 which is much better than baseline model.



---



---



## Q1.6 Last Try: Regularization and Parameter Tuning (10pt)
We have introduced features that improve the quality of our model in comparison with the first baseline. Can we do even better? After we have changed the training and test sets, it almost always makes sense to search for the optimal hyperparameters - the parameters of the model that do not change during training.

In the logistic regression that we use, the weights of each feature are changing, and we find their optimal values during training; meanwhile, the regularization parameter remains constant. This is the hyperparameter that we are going to optimize now.

We will try to beat the default parameter value $C=1$ by optimizing the regularization parameter. We will take a list of possible values of $C$ and calculate the quality metric on the validation set for each of $C$-values.

What is the value of parameter $C$ that corresponds to the highest model.


In [ ]:
# Define score variable
best_auc = 0
best_C_value = 0
# Making list of all values of C used to training
C_parameters = [0.001, 0.01, 0.1, 1, 10, 100]
# Looping over all values
for i in C_parameters:
    # Calucating score over all values of C
    auc_score = get_auc_lr_valid(X_train_with_new_features, y_train, C=i, seed=17, ratio=0.9)
    # Updated score which are best
    if auc_score > best_auc:
        best_auc = auc_score
        best_C = i
# Printing best score and C value
print("Best value of C:", best_C)
print("Auc roc score on the Valid Data with Best C value:", best_auc)

### **Discuss your findings:**

Results shows that the best Parameter C value is 0.01, which also lead to increase in roc auc score of baseline model to  0.885.

# Q2: Freeride (40pt)
**This problem can be used to present in showcase day as extra credits.**

In this part, you'll need to beat the 2 baselines mentioned in the class. No more step-by-step instructions.

Here are a few tips for finding new features: think about what you can come up with using existing features, try multiplying or dividing two of them, justify or decline your hypotheses with plots, extract useful information from time series data (time1 ... time10), do not hesitate to convert an existing feature (for example, take a logarithm), etc. We encourage you to try new ideas and models - it's fun!

In [ ]:
# Making new dataframe
new_features_df = pd.DataFrame(index=full_df.index)
# Making start month feature
new_features_df['start_month'] = (full_df['time1'].dt.year * 100 + full_df['time1'].dt.month).astype('float64')
# Making start hour feature
new_features_df['start_hour'] = full_df['time1'].dt.hour.astype('float64')
# Making start morning feature
new_features_df['morning'] = (full_df['time1'].dt.hour <= 11).astype('float64')

In [ ]:
# Plotting start_month distribution
plt.figure(figsize=(10, 6))
sns.histplot(new_features_df['start_month'], bins=30)
# Title and axis name of plot
plt.title('Distribution of Start Month')
plt.xlabel('Start Month')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Plotting start_hour distribution
plt.figure(figsize=(10, 6))
sns.histplot(new_features_df['start_hour'], bins=30)
# Title and axis name of plot
plt.title('Distribution of Start Hour')
plt.xlabel('Start Hour')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Plotting  bar plot morning
plt.figure(figsize=(8, 5))
sns.countplot(data=new_features_df, x='morning')
# Title and axis name of plot
plt.title('Distribution of Morning Sessions')
plt.xlabel('Morning')
plt.ylabel('Count')
plt.show()

In [ ]:
scaler = StandardScaler()
start_hour_scaled = scaler.fit_transform(new_features_df[['start_hour']])

# Extracting morning feature as numpy array
morning_feature = new_features_df[['morning']].values

In [ ]:
print("Shape of X_train_all_scaled:", X_train_all_scaled.shape)
print("Shape of start_hour_scaled:", start_hour_scaled.shape)
print("Shape of morning_feature:", morning_feature.shape)

In [ ]:
# Scalinng start hour daya
scaler = StandardScaler()
temp = scaler.fit_transform(new_features_df[['start_hour']])
# making morning into values
morn = new_features_df[['morning']].values
# Making new features csr matrix
X_train = csr_matrix(hstack([sprase[:idx_split,:],temp[:idx_split,:],morn[:idx_split,:]]))

In [ ]:
# Define score variable
best_auc = 0
best_C_value = 0
# Making list of all values of C used to training
C_parameters = [0.001, 0.01, 0.1, 1, 10, 100]
# Looping over all values
for i in C_parameters:
    # Calucating score over all values of C
    auc_score = get_auc_lr_valid(X_train, y_train, C=i, seed=17, ratio=0.9)
    # Updated score which are best
    if auc_score > best_auc:
        best_auc = auc_score
        best_C = i

# Printing best score and C value
print("Best value of C:", best_C)
print("Auc roc score on the Valid Data with Best C value:", best_auc)